# 🧠 Pelatihan Model Machine Learning: Klasifikasi Sampah (KNN & Ekstraksi Fitur Citra)
### Mata Kuliah: Pemrosesan Citra Digital / Visual Komputer Cerdas
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akbarlubis247/Projek-Viskom-n-citra/blob/main/training_model_colab.ipynb)

Notebook ini difokuskan khusus untuk **PROSES PEMBUATAN DAN PELATIHAN MODEL (TRAINING)** dari awal sampai jadi file model:
1. **Load Dataset Citra Sampah** (5 Kelas: Botol Kaca, Botol Plastik, Kaleng, Kardus, Kertas).
2. **Ekstraksi Fitur Citra Multi-Domain (49 Dimensi)**:
   - *Warna*: Histogram HSV (32 bin) terfokus objek + Nilai Rata-rata & Standar Deviasi CIE-LAB (6 dimensi).
   - *Bentuk*: Morfologi Kontur (Luas, Keliling, Circularity, Aspect Ratio, Solidity).
   - *Tekstur*: GLCM (Contrast, Dissimilarity, Homogeneity, Energy, Correlation) + Canny Edge Density.
3. **Normalisasi Vektor Fitur** menggunakan `StandardScaler`.
4. **Pelatihan Algoritma K-Nearest Neighbor (KNN)** dengan parameter $K=5$, *Manhattan Distance*, dan pembobotan *Distance*.
5. **Evaluasi Performa**: Akurasi, Precision, Recall, F1-Score, dan visualisasi *Confusion Matrix*.
6. **Export & Download File Model** (`knn_model.pkl` & `scaler.pkl`) yang siap digunakan di aplikasi web atau sistem nyata.

## 1. Unduh Dataset & Siapkan Lingkungan Kerja
Mendownload kode program dan dataset citra sampah dari repositori GitHub.

In [ ]:
import os

REPO_NAME = "Projek-Viskom-n-citra"
REPO_URL = "https://github.com/akbarlubis247/Projek-Viskom-n-citra.git"

if not os.path.exists(REPO_NAME) and not os.path.exists("dataset"): 
    print("📥 Mengunduh repository & dataset dari GitHub...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    print(f"🔄 Masuk ke folder {REPO_NAME}...")
    %cd {REPO_NAME}
    !git pull

# Install pustaka yang diperlukan
!pip install -q opencv-python scikit-learn scikit-image joblib matplotlib seaborn

print("\n✅ Lingkungan siap! Kategori dataset yang tersedia:")
for d in sorted(os.listdir("dataset")):
    if os.path.isdir(os.path.join("dataset", d)):
        count = len(os.listdir(os.path.join("dataset", d)))
        print(f" - {d:<15} : {count} gambar")

## 2. Definisi Modul Ekstraksi Fitur Citra (49 Dimensi)
Sistem ini mengekstrak fitur multi-domain buatan tangan (*handcrafted features*) yang merepresentasikan warna, bentuk, dan tekstur objek sampah.

In [ ]:
import cv2
import numpy as np
from skimage.feature import graycomatrix, graycoprops

def fix_mask_orientation(mask):
    """Memastikan mask objek selalu putih (255) di tengah dan background hitam (0) di tepi."""
    top, bottom = mask[0, :], mask[-1, :]
    left, right = mask[:, 0], mask[:, -1]
    border = np.concatenate([top, bottom, left, right])
    if np.mean(border > 0) > 0.5:
        mask = cv2.bitwise_not(mask)
    return mask

def preprocess_image(img, target_size=(256, 256)):
    """Resize standar 256x256, Gaussian Blur (5x5), dan Segmentasi Adaptif Otsu."""
    img_resized = cv2.resize(img, target_size)
    blurred = cv2.GaussianBlur(img_resized, (5, 5), 0)
    gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)

    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    mask = fix_mask_orientation(mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    mask_cleaned = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask_cleaned = cv2.morphologyEx(mask_cleaned, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(mask_cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    final_mask = np.zeros_like(mask_cleaned)
    if contours:
        c_largest = max(contours, key=cv2.contourArea)
        if cv2.contourArea(c_largest) > (256 * 256 * 0.02):
            cv2.drawContours(final_mask, [c_largest], -1, 255, thickness=cv2.FILLED)
        else:
            final_mask = mask_cleaned
    else:
        final_mask = mask_cleaned

    return img_resized, gray, final_mask

def extract_color_features(img_bgr, mask=None):
    """Fitur Warna: HSV Histogram 32-bin terfokus objek + Statistik LAB Mean & Std (38 Dimensi)."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    calc_mask = mask if (mask is not None and np.sum(mask > 0) > 100) else None

    h_hist = cv2.calcHist([hsv], [0], calc_mask, [16], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], calc_mask, [8], [0, 256]).flatten()
    v_hist = cv2.calcHist([hsv], [2], calc_mask, [8], [0, 256]).flatten()

    h_hist = h_hist / (np.sum(h_hist) + 1e-6)
    s_hist = s_hist / (np.sum(s_hist) + 1e-6)
    v_hist = v_hist / (np.sum(v_hist) + 1e-6)

    if calc_mask is not None:
        obj_pixels = lab[calc_mask > 0]
        mean_lab = np.mean(obj_pixels, axis=0) / 255.0
        std_lab = np.std(obj_pixels, axis=0) / 255.0
    else:
        mean_lab = np.mean(lab, axis=(0, 1)) / 255.0
        std_lab = np.std(lab, axis=(0, 1)) / 255.0

    return np.concatenate([h_hist, s_hist, v_hist, mean_lab, std_lab]).tolist()

def extract_shape_features(mask):
    """Fitur Bentuk: Luas, Keliling, Circularity, Aspect Ratio, Solidity (5 Dimensi)."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return [0.0, 0.0, 0.0, 0.0, 0.0]
    c = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(c) / (256 * 256)
    perimeter = cv2.arcLength(c, True) / 256
    circularity = (4 * np.pi * cv2.contourArea(c)) / (cv2.arcLength(c, True) ** 2) if cv2.arcLength(c, True) > 0 else 0.0
    x, y, w, h = cv2.boundingRect(c)
    aspect_ratio = float(w) / h if h > 0 else 0.0
    hull = cv2.convexHull(c)
    hull_area = cv2.contourArea(hull)
    solidity = float(cv2.contourArea(c)) / hull_area if hull_area > 0 else 0.0
    return [float(area), float(perimeter), float(circularity), float(aspect_ratio), float(solidity)]

def extract_texture_features(gray_img, mask=None):
    """Fitur Tekstur & Tepi: GLCM (5 properti) + Canny Edge Density (6 Dimensi)."""
    crop_gray = gray_img
    if mask is not None:
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            c = max(cnts, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(c)
            if w > 20 and h > 20:
                crop_gray = gray_img[y:y+h, x:x+w]
    crop_gray = cv2.resize(crop_gray, (128, 128))
    glcm = graycomatrix(crop_gray, distances=[1, 2], angles=[0, np.pi/2], levels=256, symmetric=True, normed=True)
    c = graycoprops(glcm, 'contrast').mean() / 100.0
    d = graycoprops(glcm, 'dissimilarity').mean() / 50.0
    h = graycoprops(glcm, 'homogeneity').mean()
    e = graycoprops(glcm, 'energy').mean()
    corr = graycoprops(glcm, 'correlation').mean()
    edges = cv2.Canny(crop_gray, 40, 120)
    edge_density = float(np.mean(edges > 0))
    return [float(c), float(d), float(h), float(e), float(corr), edge_density]

def extract_all_features(img_input):
    """Menggabungkan seluruh vektor fitur (Total 49 Dimensi)."""
    img_resized, gray, mask = preprocess_image(img_input)
    color_f = extract_color_features(img_resized, mask)
    shape_f = extract_shape_features(mask)
    text_f = extract_texture_features(gray, mask)
    return np.array(color_f + shape_f + text_f, dtype=np.float32), img_resized, mask

print("✅ Modul Ekstraksi Fitur 49 Dimensi berhasil dimuat!")

## 3. Ekstraksi Fitur Seluruh Dataset
Mengekstrak 49 fitur untuk setiap citra pada masing-masing kelas sampah.

In [ ]:
import glob

DATASET_DIR = "dataset"
MAX_SAMPLES_PER_CLASS = 250  # Batas sampel per kelas untuk keseimbangan & efisiensi

X = []
y = []
classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])

print(f"🚀 Memulai ekstraksi fitur untuk {len(classes)} kategori...")

for label in classes:
    folder = os.path.join(DATASET_DIR, label)
    paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        paths.extend(glob.glob(os.path.join(folder, ext)))
    
    paths.sort(key=lambda p: 0 if "user_" in os.path.basename(p) else 1)
    paths = paths[:MAX_SAMPLES_PER_CLASS]
    print(f" -> Memproses kelas '{label}': {len(paths)} gambar...")
    
    for p in paths:
        try:
            img = cv2.imread(p)
            if img is None:
                continue
            feats, _, _ = extract_all_features(img)
            X.append(feats)
            y.append(label)
        except Exception:
            pass

X = np.array(X, dtype=np.float32)
y = np.array(y)

print("\n" + "="*50)
print(f"✅ Total dataset yang diekstrak : {len(X)} sampel")
print(f"✅ Dimensi vektor fitur per citra: {X.shape[1]} fitur")
print("="*50)

## 4. Pembagian Data, Normalisasi (StandardScaler), dan Pelatihan Model KNN
- **Split Data**: 80% Data Latih (*Training*), 20% Data Uji (*Testing*).
- **StandardScaler**: Menstandarisasi skala fitur agar berdistribusi normal (mean=0, std=1).
- **K-Nearest Neighbor (KNN)**: Parameter $K=5$, pembobotan jarak (*distance weighting*), dan jarak Manhattan.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import joblib

# 1. Split Data Latih & Uji (80:20 stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"📊 Ukuran Data Latih: {len(X_train)} sampel")
print(f"📊 Ukuran Data Uji  : {len(X_test)} sampel\n")

# 2. Normalisasi Fitur dengan StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Latih Algoritma KNN (k=5, weights='distance', metric='manhattan')
print("🧠 Melatih Model K-Nearest Neighbor (KNN)...")
knn = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='manhattan')
knn.fit(X_train_scaled, y_train)

# 4. Simpan Model & Scaler ke file .pkl
os.makedirs("models", exist_ok=True)
joblib.dump(knn, "models/knn_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")

print("✅ Model berhasil dilatih dan disimpan ke:")
print("   - models/knn_model.pkl")
print("   - models/scaler.pkl")

## 5. Evaluasi Performa Model (Metrics & Confusion Matrix)
Menguji model pada data uji (*test set*) yang belum pernah dilihat saat pelatihan.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

# Prediksi pada data uji
y_pred = knn.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print("=" * 60)
print(f"🎯 AKURASI KESELURUHAN PADA DATA UJI: {acc * 100:.2f}%")
print("=" * 60)
print("\n📋 LAPORAN KLASIFIKASI LENGKAP:")
print(classification_report(y_test, y_pred))

# Visualisasi Heatmap Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=knn.classes_)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.figure.colorbar(im, ax=ax)
ax.set_xticks(np.arange(len(knn.classes_)))
ax.set_yticks(np.arange(len(knn.classes_)))
ax.set_xticklabels(knn.classes_, rotation=30, ha="right", fontsize=10)
ax.set_yticklabels(knn.classes_, fontsize=10)
ax.set_xlabel("Prediksi Model", fontweight="bold")
ax.set_ylabel("Kelas Aktual (Ground Truth)", fontweight="bold")
ax.set_title(f"Confusion Matrix Model KNN (Akurasi: {acc*100:.1f}%)", fontsize=11, fontweight="bold")

for i in range(len(knn.classes_)):
    for j in range(len(knn.classes_)):
        val = cm[i, j]
        color = "white" if val > cm.max() / 2 else "black"
        ax.text(j, i, val, ha="center", va="center", color=color, fontweight="bold")

plt.tight_layout()
plt.show()

## 6. Unduh File Model Hasil Latihan (`.pkl`)
Jalankan cell di bawah untuk langsung men-download file model `knn_model.pkl` dan `scaler.pkl` ke laptop/komputer Anda.

In [ ]:
from google.colab import files

print("📥 Mendownload file model ke perangkat lokal...")
files.download("models/knn_model.pkl")
files.download("models/scaler.pkl")
print("✅ Selesai! Model siap digunakan di aplikasi desktop atau Streamlit.")